# WIP mlflow.genai.evaluate

| 🚨 Consider a WIP until this message goes away... 🚨 |
|-|

[mlflow.genai.evaluate](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.genai.html#mlflow.genai.evaluate):

<br>

```
evaluate(
    data: 'EvaluationDatasetTypes',
    scorers: list[mlflow.genai.scorers.base.Scorer],
    predict_fn: Optional[Callable[..., Any]] = None,
    model_id: str | None = None
) -> 'EvaluationResult'
```

Evaluates the performance of a generative AI model/application using specified data and scorers:

* `data: 'EvaluationDatasetTypes'` (we'll talk about it later)
* `scorers: list[mlflow.genai.scorers.base.Scorer]` (we'll talk about it later, too)
* `predict_fn: Optional[Callable[..., Any]] = None` is for an Gen AI app
* `model_id: str | None = None` is for an AI model
* Returns `EvaluationResult`


## FIXME Jupyter Lab

🚨 **NOTE**: I really wanted to show you how to run this notebook outside Databricks using [JupyterLab: A Next-Generation Notebook Interface](https://jupyter.org/), but ran into some issues with the very first python cell and gave up...for now 🤞

<br>

```shell
uvx --with jupyterlab jupyter lab
```

In [0]:
%pip install -qU mlflow-skinny[databricks]>=3.13
%restart_python

In [0]:
import mlflow

print(f"MLflow version: {mlflow.__version__}")

## mlflow.genai

[mlflow.genai](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.genai.html)

In [0]:
from mlflow.genai import evaluate

help(mlflow.genai.evaluate)

## EvaluationDatasetTypes

[EvaluationDatasetTypes](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/evaluation/utils.py#L30-L52) cannot be imported directly (since it is guarded by [TYPE_CHECKING](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/evaluation/utils.py#L25-L52) flag).

<br>

```py
EvaluationDatasetTypes = (
    pd.DataFrame
    | pyspark.sql.dataframe.DataFrame
    | list[dict]
    | list[Trace]
    | ManagedEvaluationDataset
    | EntityEvaluationDataset
    | ConversationSimulator
    | None
)
```

<br>

An interesting part is the [try-except](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/evaluation/utils.py#L30-L52) block to `import pyspark.sql.dataframe`.
It is to check if MLflow runs in a PySpark runtime.


## Scorer

[mlflow.genai.scorers.base.Scorer](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/base.py#L218-L1120) is a [pydantic.BaseModel](https://pydantic.dev/docs/validation/latest/concepts/models/) with the following properties:

* `name` (required)
* `aggregations`
* `description`

Aggregations can be the literals or a custom function (of `int`s and `float`s):

* `_AggregationFunc: TypeAlias = Callable[[list[int | float]], float]`
* ```_AggregationType: TypeAlias = (Literal["min", "max", "mean", "median", "variance", "p90"] | _AggregationFunc)```

There are [built-in scorers](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/__init__.py) (provided by MLflow) and custom scorers.

🚨 [Report this "discrepancy"](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/__init__.py#L118).

<br>

```py
from mlflow.genai.scorers import Correctness, Safety
```

All the built-in scorers inherit from [BuiltInScorer(Judge)](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/builtin_scorers.py#L313C1-L388) class.

[Judge](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/judges/base.py#L53C1-L137) class is a...[Scorer](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/base.py#L218-L1120).

You can invoke scorers directly with a single input (for testing), or pass them to `mlflow.genai.evaluate` for running full evaluation on a dataset.


### Built-In Scorers

* [Correctness](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/builtin_scorers.py#L1686-L1878) evaluates whether the model's response supports the expected facts or response.
* [Safety](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/builtin_scorers.py#L1580-L1682) ensures that the agent's responses do not contain harmful, offensive, or toxic content.
* [a few others](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/__init__.py#L30-L50)

### FIXME Example (direct usage)

`expectations` field can only be a dict with `expected_response` or `expected_facts`.

In [0]:
import mlflow
from mlflow.genai.scorers import Correctness

scorer = Correctness(name="my_correctness")

print(f"description:\n{scorer.description}")
print("")
print(f"instructions:\n{scorer.instructions}")

In [0]:
# Scorers are callables
assessment = scorer(
    # required
    # https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/builtin_scorers.py#L1758
    inputs={
        "question": "What is the difference between reduceByKey and groupByKey in Spark?"
    },
    # required
    # https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/builtin_scorers.py#L1758
    outputs=(
        "reduceByKey aggregates data before shuffling, whereas groupByKey "
        "shuffles all data, making reduceByKey more efficient."
    ),
    # expectations: dict[str, Any] | None = None,
    expectations={
        "expected_response": (
            "reduceByKey aggregates data before shuffling"
            "groupByKey shuffles all data"
        )
    },
)
print(f"assessment:\n{assessment}")

In [0]:
import mlflow
from mlflow.genai.scorers import Correctness

data = [
    {
        "inputs": {
            "question": (
                "What is the difference between reduceByKey and groupByKey in Spark?"
            )
        },
        "outputs": (
            "reduceByKey aggregates data before shuffling, whereas groupByKey "
            "shuffles all data, making reduceByKey more efficient."
        ),
        "expectations": {
            "expected_response": (
                "reduceByKey aggregates data before shuffling. groupByKey shuffles all data"
            ),
        },
    }
]
result = mlflow.genai.evaluate(data=data, scorers=[Correctness()])
print(result)

### Correctness

[Correctness.\_\_call__](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/builtin_scorers.py#L1814-L1878)

In [0]:
help(Correctness.__call__)


`Correctness` returns a [mlflow.entities.assessment.Feedback](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/entities/assessment.py#L193-L363) (which is a [Assessment](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/entities/assessment.py#L34-L186) FWIW).

In [0]:
help(mlflow.entities.assessment.Feedback)


Among the properties of `Feedback` is `AssessmentSource` with [AssessmentSourceType](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/entities/assessment_source.py#L102C7-L194):

- `HUMAN`: Assessment performed by a human evaluator
- `LLM_JUDGE`: Assessment performed by an LLM-as-a-judge (e.g., GPT-4)
- `CODE`: Assessment performed by deterministic code/heuristics
- `SOURCE_TYPE_UNSPECIFIED`: Default when source type is not specified


## EvaluationResult

[EvaluationResult](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/evaluation/entities.py#L225-L245) is a Python `@dataclass`.